# Feature Engineering
- After the label stage, we move on feature engineer stage. In this, we gonna create the features set for our grouth truth set (both automatical and manual ones)
- There is no specific requirements for this stage so we are freely to create

# Setup
This section is used to import necessary library.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import datetime as dt
from rapidfuzz import fuzz
import re
import unicodedata

DIR_DATA = '../scopus/'
SAVE_DATA = '../dataset/'

# Load data
### Read data from scopus
- Read data of references table and bibtex table
- The bibtex table is splitted into shard for smaller size so it need to be concatenated when loaded again by pandas

In [2]:
files = sorted(Path(DIR_DATA + 'bibtex').glob("*.parquet"))

references_df = pd.read_parquet(DIR_DATA + 'references.parquet')
bibtex_df = pd.concat(
    [pd.read_parquet(file) for file in files],
    ignore_index=True
)

### Read ground truth (labelled) data
- Read the labelled data saved as csv in the previous stage

In [3]:
auto_ground_truth = pd.read_csv(SAVE_DATA + 'auto_truth.csv', dtype={'arxiv_id': 'str'})
manual_ground_truth = pd.read_csv(SAVE_DATA + 'manual_truth.csv', dtype={'arxiv_id': 'str'})

# Data exploration
- First let overview the information of bibtex dataframe and references dataframe (we have done this in labelling stage)

In [4]:
bibtex_df.info()
bibtex_df.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 181344 entries, 0 to 181343
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   arxiv_id      181344 non-null  str     
 1   citation_key  181344 non-null  str     
 2   entry_type    181344 non-null  category
 3   title         177129 non-null  str     
 4   authors       181344 non-null  object  
 5   year          180171 non-null  float64 
 6   doi           71144 non-null   str     
 7   journal       149277 non-null  str     
 8   url           41541 non-null   str     
dtypes: category(1), float64(1), object(1), str(6)
memory usage: 36.0+ MB


,arxiv_id,citation_key,entry_type,title,authors,year,doi,journal,url
0,2608.04007,shao2024deepseekmath,misc,DeepSeekMath: Pushing the Limits of Mathematic...,"[Zhihong Shao, Peiyi Wang, Qihao Zhu, Runxin X...",2024.0,NaN,NaN,NaN
1,2608.04007,hubotter2026sdpo,inproceedings,Reinforcement Learning via Self-Distillation,"[Jonas Hübotter, Frederike Lübeck, Lejs Deen B...",2026.0,NaN,Forty-third International Conference on Machin...,https://openreview.net/forum?id=QkfkxyRizZ


In [5]:
references_df.info()
references_df.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 81795 entries, 0 to 81794
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   arxiv_id             81795 non-null  str   
 1   ref_arxiv_id         81795 non-null  str   
 2   ref_title            81795 non-null  str   
 3   ref_authors          81795 non-null  object
 4   ref_submission_date  81651 non-null  str   
 5   ref_s2_id            81795 non-null  str   
dtypes: object(1), str(5)
memory usage: 14.8+ MB


,arxiv_id,ref_arxiv_id,ref_title,ref_authors,ref_submission_date,ref_s2_id
0,2608.04007,2607.05804,TurnOPD: Making On-Policy Distillation Turn-Aw...,"[Yu-Hang Zhou, Kai Zheng, Hao-Ling Li, Deng-Yu...",2026-07-07,79cee708653ca488b80b371c01510d2ca427629e
1,2608.04007,2606.19659,SAGE-OPD: Selective Agent-Guided Intervention ...,"[Yu-Hang Zhou, Li-Zhu Zhang, Yi-Fan Wu, Ming-Y...",2026-06-17,22cf5804fc22a99e5aa8abff9b280204596aca45


- Let overview the data from both ground truth

In [6]:
auto_ground_truth.info()
auto_ground_truth.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 263479 entries, 0 to 263478
Data columns (total 4 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   arxiv_id            263479 non-null  str  
 1   citation_key        263479 non-null  str  
 2   candidate_arxiv_id  263479 non-null  str  
 3   label               263479 non-null  int64
dtypes: int64(1), str(3)
memory usage: 16.9 MB


,arxiv_id,citation_key,candidate_arxiv_id,label
0,2608.06981,DBLP:conf/icml/Sohl-DicksteinW15,2606.04990,0
1,2608.06981,DBLP:conf/icml/Sohl-DicksteinW15,2605.08289,0


In [7]:
manual_ground_truth.info()
manual_ground_truth.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 2370 entries, 0 to 2369
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   arxiv_id            2370 non-null   str  
 1   citation_key        2370 non-null   str  
 2   candidate_arxiv_id  2370 non-null   str  
 3   label               2370 non-null   int64
dtypes: int64(1), str(3)
memory usage: 151.9 KB


,arxiv_id,citation_key,candidate_arxiv_id,label
0,2608.06628,kim2025train,2602.19980,0
1,2608.06628,kim2025train,2602.08676,0


- Let check with the selected arXiv ID in automatical ground truth and manual one, is there any missing in 3 specific columns in both references and bibtex table, title, authors, year, that we may need later for feature creation.
- We only check on the corresponding rows in bibtex table and reference table using merge

In [8]:
CHECK_COLS = ['title', 'authors', 'year', 'ref_title', 'ref_authors', 'ref_submission_date']

In [9]:
manual_merged = manual_ground_truth.merge(
                bibtex_df.drop(columns=['entry_type', 'doi', 'journal', 'url']),
                left_on = ['arxiv_id', 'citation_key'],
                right_on = ['arxiv_id', 'citation_key'],
                how='left'
                ).merge(
                references_df.drop(columns=['ref_s2_id']),
                left_on = ['arxiv_id', 'candidate_arxiv_id'],
                right_on = ['arxiv_id', 'ref_arxiv_id'],
                how='left'
                ).drop(columns=['ref_arxiv_id'])

In [10]:
manual_merged.info()
manual_merged.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 2370 entries, 0 to 2369
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   arxiv_id             2370 non-null   str    
 1   citation_key         2370 non-null   str    
 2   candidate_arxiv_id   2370 non-null   str    
 3   label                2370 non-null   int64  
 4   title                2370 non-null   str    
 5   authors              2370 non-null   object 
 6   year                 2370 non-null   float64
 7   ref_title            2370 non-null   str    
 8   ref_authors          2370 non-null   object 
 9   ref_submission_date  2370 non-null   str    
dtypes: float64(1), int64(1), object(2), str(6)
memory usage: 623.0+ KB


,arxiv_id,citation_key,candidate_arxiv_id,label,title,authors,year,ref_title,ref_authors,ref_submission_date
0,2608.06628,kim2025train,2602.19980,0,"Train for the worst, plan for the best: Unders...","[Jaeyeon Kim, Kulin Shah, Vasilis Kontonis, Sh...",2025.0,Discrete Diffusion Models Exploit Asymmetry to...,"[I. Trainin, Shauli Ravfogel, Omri Abend, Amir...",2026-02-23
1,2608.06628,kim2025train,2602.08676,0,"Train for the worst, plan for the best: Unders...","[Jaeyeon Kim, Kulin Shah, Vasilis Kontonis, Sh...",2025.0,LLaDA2.1: Speeding Up Text Diffusion via Token...,"[Tiwei Bie, Maosong Cao, Xiang Cao, Bingsen Ch...",2026-02-09


In [11]:
manual_check_missing = manual_merged[CHECK_COLS].isna().sum(axis=0)
print('=========== MANUAL MISSING CHECK ===========')
for col,missing in zip(CHECK_COLS, manual_check_missing):
    print(f'{col}: {missing}')

=========== MANUAL MISSING CHECK ===========
title: 0
authors: 0
year: 0
ref_title: 0
ref_authors: 0
ref_submission_date: 0


- Luckily, there is no missings information in manual ground truth
- We do the same check with automatical ground truth

In [12]:
auto_merged = auto_ground_truth.merge(
                bibtex_df.drop(columns=['entry_type', 'doi', 'journal', 'url']),
                left_on = ['arxiv_id', 'citation_key'],
                right_on = ['arxiv_id', 'citation_key'],
                how='left'
                ).merge(
                references_df.drop(columns=['ref_s2_id']),
                left_on = ['arxiv_id', 'candidate_arxiv_id'],
                right_on = ['arxiv_id', 'ref_arxiv_id'],
                how='left'
                ).drop(columns=['ref_arxiv_id'])

In [13]:
auto_merged.info()
auto_merged.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 263479 entries, 0 to 263478
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   arxiv_id             263479 non-null  str    
 1   citation_key         263479 non-null  str    
 2   candidate_arxiv_id   263479 non-null  str    
 3   label                263479 non-null  int64  
 4   title                263479 non-null  str    
 5   authors              263479 non-null  object 
 6   year                 262930 non-null  float64
 7   ref_title            263479 non-null  str    
 8   ref_authors          263479 non-null  object 
 9   ref_submission_date  262904 non-null  str    
dtypes: float64(1), int64(1), object(2), str(6)
memory usage: 67.6+ MB


,arxiv_id,citation_key,candidate_arxiv_id,label,title,authors,year,ref_title,ref_authors,ref_submission_date
0,2608.06981,DBLP:conf/icml/Sohl-DicksteinW15,2606.04990,0,Deep Unsupervised Learning using Nonequilibriu...,"[Jascha Sohl-Dickstein, Eric A. Weiss, Niru Ma...",2015.0,From Agent Traces to Trust: A Survey of Eviden...,"[Yi-Qi Wang, Jia-Qi Zhang, Tao-Tao Cai, Zi-Rui...",2026-06-03
1,2608.06981,DBLP:conf/icml/Sohl-DicksteinW15,2605.08289,0,Deep Unsupervised Learning using Nonequilibriu...,"[Jascha Sohl-Dickstein, Eric A. Weiss, Niru Ma...",2015.0,What If We Let Forecasting Forget? A Sparse Bo...,"[Fangzhao Zhang, Shi-Ming Fan, Hua Wang]",2026-05-08


In [14]:
auto_check_missing = auto_merged[CHECK_COLS].isna().sum(axis=0)
print('=========== AUTO MISSING CHECK ===========')
for col,missing in zip(CHECK_COLS, auto_check_missing):
    print(f'{col}: {missing}')

=========== AUTO MISSING CHECK ===========
title: 0
authors: 0
year: 549
ref_title: 0
ref_authors: 0
ref_submission_date: 575


- Oh, there is some missings in year and reference submission date.
- We do not fill into or remove them, we keep them to construct features later.

- The first 2 rows show that the reference submission date is followed yyyy-mm-dd pattern
- We also check all of submission dates to determine whether they all follow that patter 

In [15]:
REGEX_DATE = r'[0-9]{4}-[0-9]{2}-[0-9]{2}'
PATTERN_DATE = '%Y-%m-%d'

In [16]:
manual_ref_date = manual_merged['ref_submission_date']

format_ok_manual = manual_ref_date.str.fullmatch(
    REGEX_DATE,
    na=False
)

date_ok_manual = pd.to_datetime(
    manual_ref_date,
    format=PATTERN_DATE,
    errors='coerce'
).notna()

valid_manual = format_ok_manual & date_ok_manual
valid_manual.all()

np.True_

- Luckily, all of submission dates in manual ground truth is followed yyyy-mm-dd pattern
- We do the same thing with automatical ground truth
- There are some missing in submission dates of automatical one. We do not need to check these, so we consider these as appropriate values.

In [17]:
auto_ref_date = auto_merged['ref_submission_date']

format_ok_auto = auto_ref_date.str.fullmatch(
    REGEX_DATE,
    na=False
)

date_ok_auto = pd.to_datetime(
    auto_ref_date,
    format=PATTERN_DATE,
    errors='coerce'
).notna()

valid_auto = auto_ref_date.isna() | (format_ok_auto & date_ok_auto)
valid_auto.all()

np.True_

- Luckily, all of submission dates in automatical ground truth is followed yyyy-mm-dd pattern

# Feature selection

1. The first feature I chose is called **title ratio** which is the rate of correspondent score calculated by RapidFuzz

In [18]:
def title_ratio(title, ref_title):
    if not isinstance(title, str) or not isinstance(ref_title, str):
            return 0
    if not title or not ref_title:
        return 0.0
    return fuzz.ratio(title, ref_title)/100.

2. The second feature I chose is called **title token sort** which is the rate of comparative score when the order of words in both titles is different.

In [19]:
def title_token_sort(title, ref_title):
    if not isinstance(title, str) or not isinstance(ref_title, str):
            return 0
    if not title or not ref_title:
        return 0.0
    return fuzz.token_sort_ratio(title, ref_title)/100.

3. The third feature I chose is called **title jaccard** which is the rate of the number of words appearing in both titles on the total words in both titles.

In [20]:
def title_jaccard(title, ref_title):
    words_a = set(title.split())
    words_b = set(ref_title.split())

    if not words_a or not words_b:
        return 0

    return len(words_a & words_b)/len(words_a | words_b)

4. The forth feature I chose is called **title length ratio** which is the rate of the length of longer title on length of shorter title (length here is the number of characters in that title)

In [21]:
def title_length_ratio(title, ref_title):
    len_a = len(title)
    len_b = len(ref_title)

    if len_a == 0 or len_b == 0:
        return 0

    return min(len_a, len_b)/max(len_a, len_b) 

5. The fifth feature I choose is called **author overlap** which is the rate of the number of matched authors on the length of shorter author list

In [22]:
def author_overlap(authors, ref_authors):
    set_authors_1 = set(name for name in authors if name)
    set_authors_2 = set(name for name in ref_authors if name)

    if not set_authors_1 or not set_authors_2:
            return 0
    return len(set_authors_1 & set_authors_2)/min(len(set_authors_1),len(set_authors_2))

6. The sixth feature I choose is called **first author similarity** which is the rate of correspondent score between two first author namnes.

In [23]:
def first_author_similarity(authors, ref_authors):
    
    valid_types = (list, tuple, np.ndarray)

    if not isinstance(authors, valid_types):
        return 0.0
    if not isinstance(ref_authors, valid_types):
        return 0.0

    if len(authors) == 0 or len(ref_authors) == 0:
        return 0.0

    if not authors[0] or not ref_authors[0]:
        return 0.0

    return fuzz.ratio(authors[0], ref_authors[0]) / 100

7. The seventh feature I choose is called **year gap** which is the absolute difference of two reference years.

In [24]:
def year_gap(year, ref_year):
    if pd.isna(year) or pd.isna(ref_year):
        return np.nan
    return abs(year-ref_year)

8. The eighth feature I choose is called **title  missing** which confirms whether it has missing in either bibtex title or reference title.

In [25]:
def title_missing(title, ref_title):
    return int(not title or not ref_title)

9. The ninth feature I choose is called **author  missing** which confirms whether it has missing in either bibtex authors or reference authors.

In [26]:
def author_missing(authors, ref_authors):
    return int(not any(authors) or not any(ref_authors))

10. The tenth feature I choose is called **year  missing** which confirms whether it has missing in either bibtex year or reference year.

In [27]:
def year_missing(year, ref_year):
    return int(pd.isna(year) or pd.isna(ref_year))

# Preprocessing and clean neccessary data

- The title or author name should be normalized before the feature creating from them.

In [28]:
def normalize_text(value):
    if not isinstance(value, str):
        return ""

    value = unicodedata.normalize("NFC", value)
    value = value.casefold()
    value = re.sub(r"\s+", " ", value).strip()
    value = value.rstrip(".,").strip()
    return value

- The year from the submission date must be extracted
- The date is followed the yyyy-mm-dd pattern as above exploration

In [29]:
def extract_year (ref_submission_date):
    if pd.isna(ref_submission_date):
        return pd.NA
    return int(ref_submission_date[:4])

# Feature construction

- We gonna construct features for manual ground truth

In [30]:
manual_dataset_lst = []

for _, row in manual_merged.iterrows():
    normalized_title = normalize_text(row['title'])
    normalized_ref_title = normalize_text(row['ref_title'])
    normalized_authors = [normalize_text(author) for author in row['authors']]
    normailize_ref_authors = [normalize_text(author) for author in row['ref_authors']]
    year = int(row['year']) if not pd.isna(row['year']) else pd.NA
    ref_year = extract_year(row['ref_submission_date'])

    manual_dataset_lst.append({
        'arxiv_id': row['arxiv_id'],
        'citation_key': row['citation_key'],
        'candidate_arxiv_id': row['candidate_arxiv_id'],
        'label': row['label'],

        'title_ratio': title_ratio(normalized_title, normalized_ref_title),
        'title_token_sort': title_token_sort(normalized_title, normalized_ref_title),
        'title_jaccard': title_jaccard(normalized_title, normalized_ref_title),
        'title_length_ratio': title_length_ratio(normalized_title, normalized_ref_title),
        'author_overlap': author_overlap(normalized_authors, normailize_ref_authors),
        'first_author_similarity': first_author_similarity(normalized_authors, normailize_ref_authors),
        'year_gap': year_gap(year, ref_year),
        'title_missing': title_missing(normalized_title, normalized_ref_title),
        'author_missing': author_missing(normalized_authors, normailize_ref_authors),
        'year_missing': year_missing(year, ref_year)
    })
manual_dataset_df = pd.DataFrame(manual_dataset_lst)

In [31]:
manual_dataset_df.info()
manual_dataset_df.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 2370 entries, 0 to 2369
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   arxiv_id                 2370 non-null   str    
 1   citation_key             2370 non-null   str    
 2   candidate_arxiv_id       2370 non-null   str    
 3   label                    2370 non-null   int64  
 4   title_ratio              2370 non-null   float64
 5   title_token_sort         2370 non-null   float64
 6   title_jaccard            2370 non-null   float64
 7   title_length_ratio       2370 non-null   float64
 8   author_overlap           2370 non-null   float64
 9   first_author_similarity  2370 non-null   float64
 10  year_gap                 2370 non-null   int64  
 11  title_missing            2370 non-null   int64  
 12  author_missing           2370 non-null   int64  
 13  year_missing             2370 non-null   int64  
dtypes: float64(6), int64(5), str(3)
mem

,arxiv_id,citation_key,candidate_arxiv_id,label,title_ratio,title_token_sort,title_jaccard,title_length_ratio,author_overlap,first_author_similarity,year_gap,title_missing,author_missing,year_missing
0,2608.06628,kim2025train,2602.19980,0,0.373494,0.469880,0.000000,0.865169,0.0,0.285714,1,0,0,0
1,2608.06628,kim2025train,2602.08676,0,0.377622,0.461538,0.052632,0.606742,0.0,0.300000,1,0,0,0


- We gonna construct features for automatical ground truth

In [32]:
auto_dataset_lst = []

for _, row in auto_merged.iterrows():
    normalized_title = normalize_text(row['title'])
    normalized_ref_title = normalize_text(row['ref_title'])
    normalized_authors = [normalize_text(author) for author in row['authors']]
    normailize_ref_authors = [normalize_text(author) for author in row['ref_authors']]
    year = int(row['year']) if not pd.isna(row['year']) else pd.NA
    ref_year = extract_year(row['ref_submission_date'])

    auto_dataset_lst.append({
        'arxiv_id': row['arxiv_id'],
        'citation_key': row['citation_key'],
        'candidate_arxiv_id': row['candidate_arxiv_id'],
        'label': row['label'],

        'title_ratio': title_ratio(normalized_title, normalized_ref_title),
        'title_token_sort': title_token_sort(normalized_title, normalized_ref_title),
        'title_jaccard': title_jaccard(normalized_title, normalized_ref_title),
        'title_length_ratio': title_length_ratio(normalized_title, normalized_ref_title),
        'author_overlap': author_overlap(normalized_authors, normailize_ref_authors),
        'first_author_similarity': first_author_similarity(normalized_authors, normailize_ref_authors),
        'year_gap': year_gap(year, ref_year),
        'title_missing': title_missing(normalized_title, normalized_ref_title),
        'author_missing': author_missing(normalized_authors, normailize_ref_authors),
        'year_missing': year_missing(year, ref_year)
    })
auto_dataset_df = pd.DataFrame(auto_dataset_lst)

In [33]:
auto_dataset_df.info()
auto_dataset_df.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 263479 entries, 0 to 263478
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   arxiv_id                 263479 non-null  str    
 1   citation_key             263479 non-null  str    
 2   candidate_arxiv_id       263479 non-null  str    
 3   label                    263479 non-null  int64  
 4   title_ratio              263479 non-null  float64
 5   title_token_sort         263479 non-null  float64
 6   title_jaccard            263479 non-null  float64
 7   title_length_ratio       263479 non-null  float64
 8   author_overlap           263479 non-null  float64
 9   first_author_similarity  263479 non-null  float64
 10  year_gap                 262355 non-null  float64
 11  title_missing            263479 non-null  int64  
 12  author_missing           263479 non-null  int64  
 13  year_missing             263479 non-null  int64  
dtypes: float64(7), 

,arxiv_id,citation_key,candidate_arxiv_id,label,title_ratio,title_token_sort,title_jaccard,title_length_ratio,author_overlap,first_author_similarity,year_gap,title_missing,author_missing,year_missing
0,2608.06981,DBLP:conf/icml/Sohl-DicksteinW15,2606.04990,0,0.382166,0.394904,0.0,0.652632,0.0,0.193548,11.0,0,0,0
1,2608.06981,DBLP:conf/icml/Sohl-DicksteinW15,2605.08289,0,0.324324,0.297297,0.0,0.720930,0.0,0.342857,11.0,0,0,0


# Split Dataset
- The requirements are:
  1. Test set: 1 manually labelled publication + 1 automatically labelled publication
  2. Valid set: 1 manually labelled publication + 1 automatically labelled publication
  3. Train set: remaining labelled publications

In [34]:
rng = np.random.default_rng(42)

- We choose test set randomly from remaining pool

In [35]:
test_manual_id = rng.choice(manual_dataset_df["arxiv_id"].unique())
test_auto_id = rng.choice(auto_dataset_df["arxiv_id"].unique())

test_manual_publication = manual_dataset_df.loc[
    manual_dataset_df["arxiv_id"] == test_manual_id
]

test_auto_publication = auto_dataset_df.loc[
    auto_dataset_df["arxiv_id"] == test_auto_id
]
print (f'{test_manual_id}: {len(test_manual_publication)} entries')
print (f'{test_auto_id}: {len(test_auto_publication)} entries')

2608.06628: 484 entries
2608.08265: 361 entries


In [36]:
test_set = pd.concat([test_manual_publication, test_auto_publication], ignore_index=True)
test_set.info()
test_set.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   arxiv_id                 845 non-null    str    
 1   citation_key             845 non-null    str    
 2   candidate_arxiv_id       845 non-null    str    
 3   label                    845 non-null    int64  
 4   title_ratio              845 non-null    float64
 5   title_token_sort         845 non-null    float64
 6   title_jaccard            845 non-null    float64
 7   title_length_ratio       845 non-null    float64
 8   author_overlap           845 non-null    float64
 9   first_author_similarity  845 non-null    float64
 10  year_gap                 826 non-null    float64
 11  title_missing            845 non-null    int64  
 12  author_missing           845 non-null    int64  
 13  year_missing             845 non-null    int64  
dtypes: float64(7), int64(4), str(3)
memor

,arxiv_id,citation_key,candidate_arxiv_id,label,title_ratio,title_token_sort,title_jaccard,title_length_ratio,author_overlap,first_author_similarity,year_gap,title_missing,author_missing,year_missing
0,2608.06628,kim2025train,2602.19980,0,0.373494,0.469880,0.000000,0.865169,0.0,0.285714,1.0,0,0,0
1,2608.06628,kim2025train,2602.08676,0,0.377622,0.461538,0.052632,0.606742,0.0,0.300000,1.0,0,0,0


- We choose valid set randomly from remaining pool

In [37]:
manual_remaining_valid = manual_dataset_df.loc[manual_dataset_df["arxiv_id"] != test_manual_id]
auto_remaining_valid = auto_dataset_df.loc[auto_dataset_df["arxiv_id"] != test_auto_id]

valid_manual_id = rng.choice(manual_remaining_valid["arxiv_id"].unique())
valid_auto_id = rng.choice(auto_remaining_valid["arxiv_id"].unique())

valid_manual_publication = manual_remaining_valid.loc[
    manual_remaining_valid["arxiv_id"] == valid_manual_id
]

valid_auto_publication = auto_remaining_valid.loc[
    auto_remaining_valid["arxiv_id"] == valid_auto_id
]
print (f'{valid_manual_id}: {len(valid_manual_publication)} entries')
print (f'{valid_auto_id}: {len(valid_auto_publication)} entries')

2608.06791: 484 entries
2608.07969: 100 entries


In [38]:
valid_set = pd.concat([valid_manual_publication, valid_auto_publication], ignore_index=True)
valid_set.info()
valid_set.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 584 entries, 0 to 583
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   arxiv_id                 584 non-null    str    
 1   citation_key             584 non-null    str    
 2   candidate_arxiv_id       584 non-null    str    
 3   label                    584 non-null    int64  
 4   title_ratio              584 non-null    float64
 5   title_token_sort         584 non-null    float64
 6   title_jaccard            584 non-null    float64
 7   title_length_ratio       584 non-null    float64
 8   author_overlap           584 non-null    float64
 9   first_author_similarity  584 non-null    float64
 10  year_gap                 584 non-null    float64
 11  title_missing            584 non-null    int64  
 12  author_missing           584 non-null    int64  
 13  year_missing             584 non-null    int64  
dtypes: float64(7), int64(4), str(3)
memor

,arxiv_id,citation_key,candidate_arxiv_id,label,title_ratio,title_token_sort,title_jaccard,title_length_ratio,author_overlap,first_author_similarity,year_gap,title_missing,author_missing,year_missing
0,2608.06791,cong2018besteffort,2606.30949,0,0.359712,0.417266,0.0,0.737500,0.25,0.444444,8.0,0,0,0
1,2608.06791,cong2018besteffort,2605.13536,0,0.374269,0.374269,0.0,0.526786,0.00,0.272727,8.0,0,0,0


- The remaining pool is train set

In [39]:
manual_remaining_train = manual_remaining_valid.loc[manual_remaining_valid["arxiv_id"] != valid_manual_id]
auto_remaining_train = auto_remaining_valid.loc[auto_remaining_valid["arxiv_id"] != valid_auto_id]

In [40]:
train_set = pd.concat([manual_remaining_train,auto_remaining_train], ignore_index=True)
train_set.to_csv(SAVE_DATA + 'train_set.csv', index=False)
train_set.info()
train_set.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 264420 entries, 0 to 264419
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   arxiv_id                 264420 non-null  str    
 1   citation_key             264420 non-null  str    
 2   candidate_arxiv_id       264420 non-null  str    
 3   label                    264420 non-null  int64  
 4   title_ratio              264420 non-null  float64
 5   title_token_sort         264420 non-null  float64
 6   title_jaccard            264420 non-null  float64
 7   title_length_ratio       264420 non-null  float64
 8   author_overlap           264420 non-null  float64
 9   first_author_similarity  264420 non-null  float64
 10  year_gap                 263315 non-null  float64
 11  title_missing            264420 non-null  int64  
 12  author_missing           264420 non-null  int64  
 13  year_missing             264420 non-null  int64  
dtypes: float64(7), 

,arxiv_id,citation_key,candidate_arxiv_id,label,title_ratio,title_token_sort,title_jaccard,title_length_ratio,author_overlap,first_author_similarity,year_gap,title_missing,author_missing,year_missing
0,2608.06434,pi0,2507.01424,0,0.659686,0.607330,0.437500,0.578512,0.0,0.333333,1.0,0,0,0
1,2608.06434,pi0,2506.01953,0,0.392638,0.404908,0.111111,0.752688,0.0,0.210526,1.0,0,0,0


- Check to ensure that data leakage is not existent

In [41]:
train_ids = set(train_set["arxiv_id"])
valid_ids = set(valid_set["arxiv_id"])
test_ids = set(test_set["arxiv_id"])

assert train_ids.isdisjoint(valid_ids)
assert train_ids.isdisjoint(test_ids)
assert valid_ids.isdisjoint(test_ids)

- We save them into csv files for later model training

In [42]:
test_set.to_csv(SAVE_DATA + 'test_set.csv', index=False)
valid_set.to_csv(SAVE_DATA + 'valid_set.csv', index=False)
train_set.to_csv(SAVE_DATA + 'train_set.csv', index=False)

We have done feature engineering and split dataset here !!